# 05 · Next-Month Cash-Flow Risk
- **Expected income:** median of the last 3 months for stable earners, mean of the last 6 months for volatile earners.
- **Expected outflows:** the recurring median plus the weighted recent variable spend, multiplied by the seasonal index, plus known seasonal one-offs.
- **Expected ending balance:** balance + expected income − expected outflows.
- **Uncertainty:** the customer's own month-to-month volatility.
- **Deficit probability:** the chance that the ending balance falls below a **safety cushion** of 25% of expected outflows. Income and bills arrive on different days, so a thin positive month-end still means the account runs dry mid-month.

Full assumptions and limitations are in `docs/cashflow_risk_methodology.md`.

In [1]:
import sys, json
from pathlib import Path
import pandas as pd, numpy as np
from IPython.display import Image, display, Markdown
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [2]:
fc = pd.read_csv(ROOT / "data/analytics/cashflow_forecast.csv")
print("Forecast month:", fc.forecast_month.iloc[0], "| seasonal factor:", fc.seasonal_factor_used.iloc[0])
fc["cashflow_risk_tier"].value_counts()

Forecast month: 2025-10 | seasonal factor: 1.294


cashflow_risk_tier
Low       111
High        6
Medium      3
Name: count, dtype: int64

In [3]:
fc.sort_values("deficit_probability", ascending=False).head(10)[["customer_id","current_balance","expected_income","expected_outflows","expected_ending_balance","safety_cushion","deficit_probability","cashflow_risk_tier"]]

,customer_id,current_balance,expected_income,expected_outflows,expected_ending_balance,safety_cushion,deficit_probability,cashflow_risk_tier
27,C0028,"3,249.27","13,445.90","18,224.69","-1,529.52","4,556.17",1.00,High
60,C0061,"1,775.27","7,812.34","13,835.77","-4,248.15","3,458.94",1.00,High
84,C0085,"11,097.79","51,261.00","65,331.45","-2,972.66","16,332.86",0.97,High
18,C0019,"14,316.01","60,659.12","88,197.11","-13,221.98","22,049.28",0.95,High
93,C0094,"13,085.81","66,974.00","68,468.28","11,591.53","17,117.07",0.95,High
20,C0021,"5,187.88","10,846.19","14,302.48","1,731.59","3,575.62",0.81,High
75,C0076,"26,393.90","75,638.00","77,642.61","24,389.29","19,410.65",0.37,Medium
78,C0079,"8,810.55","12,432.12","15,636.29","5,606.37","3,909.07",0.23,Medium
79,C0080,"28,476.60","35,125.82","39,644.14","23,958.28","9,911.03",0.20,Medium
57,C0058,"63,737.93","46,826.44","44,400.63","66,163.74","11,100.16",0.17,Low


## Watch-list from SQL (Q20)

In [4]:
pd.read_csv(ROOT / "sql/results/Q20_next_month_risk_watch_list_score_forecas.csv")

,customer_id,occupation_type,city,stress_score,stress_band,top_stress_driver,current_balance,expected_net_cash_flow,expected_ending_balance,deficit_probability_pct,cashflow_risk_tier,recommended_action
0,C0061,Student,Kolkata,57.30,Moderate,savings_margin,1775,-6023,-4248,100.00,High,Contact this week: offer payment-date realignm...
1,C0028,Student,Nagpur,53.40,Moderate,savings_margin,3249,-4779,-1530,100.00,High,Contact this week: offer payment-date realignm...
2,C0085,Salaried,Hyderabad,82.50,Critical,savings_margin,11098,-14070,-2973,96.60,High,Contact this week: offer payment-date realignm...
3,C0019,Freelancer/Gig,Patna,77.50,High,savings_margin,14316,-27538,-13222,95.40,High,Contact this week: offer payment-date realignm...
4,C0094,Salaried,Chennai,58.30,Moderate,fixed_obligations,13086,-1494,11592,95.10,High,Contact this week: offer payment-date realignm...
5,C0021,Student,Mumbai,53.80,Moderate,savings_margin,5188,-3456,1732,81.20,High,Contact this week: offer payment-date realignm...
6,C0076,Salaried,Indore,74.50,High,emergency_buffer,26394,-2005,24389,36.90,Medium,Proactive nudge: spending alert + budget plan
7,C0079,Student,Bengaluru,54.90,Moderate,savings_margin,8811,-3204,5606,22.60,Medium,Monitor: low-balance alert
8,C0080,Freelancer/Gig,Kolkata,46.60,Moderate,emergency_buffer,28477,-4518,23958,20.50,Medium,Monitor: low-balance alert
